## tl;dr

- 扣除基准交易成本后，对全收益指数年化超额：IF **-0.31%**、IH **-0.44%**、IC **10.03%**、IM **8.20%**。
- 四个品种分别从各自上市日起独立复利，不构造四品种每日等权组合。
- 结果证明历史主力路径存在可观贴水收益，但不是全合约最优选择回测；缺少新旧合约同刻报价、盘口和真实成交回报，结论应视为研究级估计。

## Context & Methods

目标是测算做多四类股指期货主力合约并持续展期的实际净收益。模型取 15:00 对齐价，合约不变时使用真实期货收益；主连换码日用现货收益替代不可交易的主连跳变。

### Key Assumptions

- 1 倍名义敞口；现金收益 0%。
- 手续费为成交金额 0.000023，每边滑点 0.2 指数点。
- 开仓、换月两边和期末平仓都计成本。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
OUT = ROOT / 'outputs'
summary = pd.read_csv(OUT / 'summary.csv')
annual = pd.read_csv(OUT / 'annual_returns.csv')
daily = pd.read_csv(OUT / 'daily_returns.csv', parse_dates=['date'])
sensitivity = pd.read_csv(OUT / 'sensitivity.csv')

## Data

检查样本期、对齐天数、空值和主力换月次数。

In [ ]:
quality = daily.groupby('product').agg(
    start=('date','min'), end=('date','max'), rows=('date','size'),
    roll_count=('is_roll','sum'), null_futures=('futures_close',lambda x:x.isna().sum()),
    null_spot=('spot_close',lambda x:x.isna().sum()),
    null_benchmark=('benchmark_close',lambda x:x.isna().sum()))
quality

## Results

全样本收益、风险、贴水比例与成本汇总。

In [ ]:
cols = ['product','start_date','end_date','strategy_cagr','benchmark_cagr',
        'annualized_excess_vs_benchmark','strategy_max_drawdown','tracking_error',
        'roll_count','median_basis_pct','discount_day_share','total_transaction_cost']
summary[cols].style.format({c:'{:.2%}' for c in ['strategy_cagr','benchmark_cagr',
 'annualized_excess_vs_benchmark','strategy_max_drawdown','tracking_error','median_basis_pct',
 'discount_day_share','total_transaction_cost']})

In [ ]:
colors={'IF':'#2475B0','IH':'#D6A21E','IC':'#E6812F','IM':'#7A8F35'}
fig, axes = plt.subplots(2,2,figsize=(14,9))
for ax,(p,g) in zip(axes.flat,daily.groupby('product',sort=False)):
    ax.plot(g.date,g.strategy_wealth,label=f'{p}期货策略',color=colors[p])
    ax.plot(g.date,g.benchmark_wealth,label='全收益基准',color='#6B7280',ls='--')
    ax.set_title(f'{p}独立收益测试'); ax.grid(axis='y',alpha=.25); ax.legend(frameon=False)
fig.tight_layout(); plt.show()

In [ ]:
pivot=annual.pivot(index='year',columns='product',values='excess_return')
pivot.style.format('{:.2%}').background_gradient(cmap='RdYlBu',axis=None,vmin=-.15,vmax=.30)

### 成本与现金收益敏感性

保守成本为 2 倍交易费率和每边 0.4 点滑点；现金情景只对未占用保证金的 88% 资金计 2% 年收益。

In [ ]:
sensitivity.pivot(index='product',columns='scenario',values='annualized_excess_vs_benchmark').style.format('{:.2%}')

## Takeaways

- IC、IM 的历史贴水收益最强，但也最依赖时期与合约选择。
- IF 在 2010—2014 年多次出现负超额，说明滚贴水不是稳定无风险利差。
- 生产版本应补充全合约同刻行情，以固定、无前视的换月规则重跑，并用真实成交回报替换滑点假设。